# 서버 과부하 실측

대기열 모형과 로컬 HTTP 응답 시간을 대조하고 재시도와 부하 차단이 처리량과 꼬리 지연에 미치는 영향을 측정합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/server_overload.ipynb)

Colab 런타임은 **CPU**를 선택하고 위에서부터 실행하세요. 첫 코드 셀의 `QUICK = True`는 축소 조건을 사용합니다.

In [ ]:
QUICK = False

import asyncio
from collections import defaultdict, deque
import heapq
from http.server import BaseHTTPRequestHandler, HTTPServer
import math
import random
from statistics import fmean
from threading import Thread
import time


SEED = 20261119


def heading(number, title):
    print(f"\n## 실험{number} {title}", flush=True)


def verdict(passed):
    return "PASS" if passed else "FAIL"


def result(passed, rows, **details):
    print(f"판정 {verdict(passed)}", flush=True)
    return {"passed": passed, "rows": rows, **details}


def percentile(sorted_values, fraction):
    """오름차순 표본의 최근접 순위 백분위수."""
    return sorted_values[math.ceil(len(sorted_values) * fraction) - 1]

### 그래프 언어

아래 셀에서 `CHART_LANG`의 기본값을 고르세요. `ko`는 한국어, `en`은 영어, `both`는 각 그래프를 한국어와 영어 순서로 그립니다. 한글 글꼴은 필요할 때 내려받습니다.

In [ ]:
# Chart language and style
import os
import platform
import sys
from pathlib import Path
from urllib.request import urlopen

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text

CHART_LANG = os.environ.get('CHART_LANG', 'ko')
CHART_COLORS = ('#2a78d6', '#eb6834', '#1baf7a')
CHART_TEXT = '#0b0b0b'
CHART_MUTED = '#52514e'
CHART_GRID = '#e4e3df'
CHART_BG = '#fcfcfb'
CHART_LIMIT = '#e34948'

in_colab = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
chart_font = matplotlib.rcParamsDefault['font.family']
requested_language = CHART_LANG
if requested_language not in ('ko', 'en', 'both'):
    raise ValueError("CHART_LANG must be ko, en or both")
if requested_language == 'both':
    CHART_LANG = 'ko'
if CHART_LANG == 'ko':
    # NanumGothic is licensed under the SIL OFL 1.1.
    nanum = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    try:
        if not nanum.is_file():
            nanum = Path('fonts/NanumGothic-Regular.ttf')
            if not nanum.is_file():
                nanum.parent.mkdir(exist_ok=True)
                url = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
                with urlopen(url, timeout=60) as response:
                    nanum.write_bytes(response.read())
        font_manager.fontManager.addfont(str(nanum))
        chart_font = font_manager.FontProperties(fname=str(nanum)).get_name()
    except Exception as exc:
        CHART_LANG = 'en'
        reason = ' '.join(str(exc).splitlines()) or type(exc).__name__
        print(f'차트 언어 en, 한글 글꼴 설정 실패, {reason}')
    else:
        print(f'차트 언어 ko, 글꼴 {chart_font}')
else:
    CHART_LANG = 'en'

matplotlib.rcParams.update({
    'figure.figsize': (8, 4.5),
    'figure.dpi': 200,
    'savefig.dpi': 200,
    'figure.facecolor': CHART_BG,
    'savefig.facecolor': CHART_BG,
    'axes.facecolor': CHART_BG,
    'axes.edgecolor': CHART_MUTED,
    'axes.labelcolor': CHART_TEXT,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
    'axes.unicode_minus': False,
    'axes.prop_cycle': matplotlib.rcsetup.cycler(color=CHART_COLORS),
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': CHART_GRID,
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'text.color': CHART_TEXT,
    'xtick.color': CHART_MUTED,
    'ytick.color': CHART_MUTED,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'legend.frameon': False,
    'font.family': chart_font,
})

chart_location = 'Colab' if in_colab else platform.system()
CPU_RUNTIME = f'{chart_location} CPU ({platform.machine()}), Python {platform.python_version()}'


def finish_chart(fig, runtime=CPU_RUNTIME, top=1):
    for text in fig.findobj(Text):
        label = text.get_text()
        if '·' in label:
            text.set_text(label.replace(' ·', ',').replace('·', ','))
    runtime = runtime.replace(' ·', ',').replace('·', ',')
    fig.tight_layout(rect=(0, 0.065, 1, top))
    fig.text(0.02, 0.025, runtime, fontsize=8, color=CHART_MUTED)
    plt.show()
if requested_language == 'both' and CHART_LANG == 'ko':
    CHART_LANG = 'both'
chart_languages = ('ko', 'en') if CHART_LANG == 'both' else (CHART_LANG,)

### 차트 표기

응답 시간은 밀리초, 처리량은 초당 완료 건수로 읽습니다.

In [ ]:
labels = {
    'ko': {
        'response': '사용률별 평균 응답 시간',
        'util': '목표 사용률 (%)',
        'ms': '평균 응답 시간 (ms)',
        'theory': 'M/M/1 이론',
        'sim': '시뮬레이션',
        'http': '로컬 HTTP',
        'response_note': '처리율 100건/초, 사용률별 시뮬레이션 {n:,}건, HTTP {s}초',
        'retry': '재시도 정책별 백엔드 요청 수',
        'seconds': '경과 시간 (초)',
        'attempts': '백엔드 요청 (건/초)',
        'unlimited': '무제한 재시도',
        'backoff': '3회 + 백오프',
        'budget_only': '재시도 예산',
        'retry_note': '신규 10,100건/초, 점선은 처리 한계 10,000건/초',
        'throughput_qps': '대기열 상한과 처리량',
        'qps': '구간 내 완료 (건/초)',
        'p99_ms': '대기열 상한과 수락 요청의 p99',
        'p99': '응답 시간 p99 (ms)',
        'queue': '무제한 대기열',
        'shed': '상한 5건 + 503',
        'shed_note': '작업자 10개, 도착률 120건/초, 동일한 도착열 {s}초',
    },
    'en': {
        'response': 'Mean response time by utilization',
        'util': 'Target utilization (%)',
        'ms': 'Mean response time (ms)',
        'theory': 'M/M/1 theory',
        'sim': 'Simulation',
        'http': 'Local HTTP',
        'response_note': 'Service rate 100/s, {n:,} simulated per load, HTTP {s} s',
        'retry': 'Backend attempts by retry policy',
        'seconds': 'Elapsed time (s)',
        'attempts': 'Backend attempts per second',
        'unlimited': 'Unlimited retries',
        'backoff': '3 retries + backoff',
        'budget_only': 'Retry budget',
        'retry_note': '10,100 new requests/s, dashed line at capacity 10,000/s',
        'throughput_qps': 'Throughput by queue policy',
        'qps': 'Completions per second',
        'p99_ms': 'Accepted-request p99 by queue policy',
        'p99': 'p99 response time (ms)',
        'queue': 'Unlimited queue',
        'shed': 'Queue limit 5 + HTTP 503',
        'shed_note': '10 workers, arrival rate 120/s, same {s} s arrival trace',
    },
}

### 1. M/M/1 대기열

한 작업자에 요청이 포아송 과정으로 도착하고 처리 시간이 지수분포를 따릅니다. 정상 상태에서 출발한 표본 평균과 이론값을 비교합니다.

In [ ]:
def experiment1(quick=False):
    heading(1, "M/M/1 평균 응답 시간")
    mu = 100
    paths = 2000
    per_path = 20 if quick else 100
    print(f"처리율 {mu}/초, 도착률마다 {paths * per_path:,}건")
    print("도착/초    측정 ms    이론 ms    상대 오차    판정")
    rows = []
    for rate in (50, 70, 90, 95, 99):
        rng = random.Random(SEED + rate)
        rho = rate / mu
        total = 0.0
        for stratum in range(paths):
            # 정상 상태의 도착 직전 고객 수는 기하분포다.
            # 분위 구간마다 한 번 시작해 혼잡한 초기 상태도 표집한다.
            u = (stratum + rng.random()) / paths
            ahead = math.floor(math.log1p(-u) / math.log(rho))
            # 서비스 중인 고객의 잔여 시간도 지수분포다.
            finish = rng.gammavariate(ahead, 1 / mu) if ahead else 0.0
            arrival = 0.0
            for i in range(per_path):
                if i:
                    arrival += rng.expovariate(rate)
                finish = max(arrival, finish) + rng.expovariate(mu)
                total += finish - arrival
        mean = total / (paths * per_path)
        theory = 1 / (mu - rate)
        error = abs(mean - theory) / theory
        rows.append(dict(rate_qps=rate, samples=paths * per_path,
                         mean_ms=1000 * mean, theory_ms=1000 * theory,
                         relative_error=error, passed=error <= 0.05))
        print(f"{rate:7} {mean * 1000:10.2f} {theory * 1000:10.2f}"
              f" {error:11.2%}    {verdict(error <= 0.05)}")
    return result(all(row["passed"] for row in rows), rows, paths=paths)


simulation = experiment1(QUICK)

### 2. HTTP 응답 시간

같은 평균 처리율의 HTTP 서버를 루프백 스레드에서 띄웁니다. 예정 도착부터 본문 수신까지 잰 뒤 성공·실패 건수와 성공 요청의 지연을 표시합니다.

In [ ]:
class LocalServer(HTTPServer):
    # 운영체제의 연결 대기열. HTTPServer 자체가 작업자 한 개다.
    request_queue_size = 256


class SleepHandler(BaseHTTPRequestHandler):
    timeout = 1  # 연결 후 헤더가 끝나지 않아도 작업자를 반환한다.

    def do_GET(self):
        time.sleep(self.server.service_rng.expovariate(100))
        try:
            self.send_response(200)
            self.send_header("Content-Length", "2")
            self.end_headers()
            self.wfile.write(b"OK")
        except OSError:
            # 배출 기한에 취소된 클라이언트는 이미 실패로 집계된다.
            pass

    def log_message(self, format, *args):
        pass


async def fetch_local(port, scheduled):
    loop = asyncio.get_running_loop()
    writer = None
    try:
        reader, writer = await asyncio.open_connection("127.0.0.1", port)
        writer.write(b"GET / HTTP/1.0\r\nHost: localhost\r\n\r\n")
        await writer.drain()
        status = await reader.readline()
        await reader.readuntil(b"\r\n\r\n")
        body = await reader.readexactly(2)
        if status.split()[1:2] == [b"200"] and body == b"OK":
            return (loop.time() - scheduled) * 1000
    except (OSError, EOFError, asyncio.LimitOverrunError):
        pass
    finally:
        if writer is not None:
            writer.close()
    return None


async def http_window(port, rate, duration, rng):
    loop = asyncio.get_running_loop()
    start = loop.time()
    end = start + duration
    scheduled = start + rng.expovariate(rate)
    tasks = []
    missed = 0
    while scheduled < end:
        await asyncio.sleep(max(0, scheduled - loop.time()))
        if loop.time() < end:
            tasks.append(asyncio.create_task(fetch_local(port, scheduled)))
        else:
            missed += 1
        scheduled += rng.expovariate(rate)

    await asyncio.sleep(max(0, end - loop.time()))
    if tasks:
        _, pending = await asyncio.wait(tasks, timeout=3)
        for task in pending:
            task.cancel()
    outcomes = await asyncio.gather(*tasks, return_exceptions=True)
    samples = sorted(value for value in outcomes if isinstance(value, float))
    return len(tasks) + missed, missed, samples


async def experiment2(quick=False):
    heading(2, "HTTP 작업자 한 개의 응답 시간")
    duration = 5 if quick else 20
    print(f"도착 구간 {duration}초, 예정 도착부터 본문 수신까지 측정")
    print("이론은 평균값. 실제 값에는 HTTP·스케줄링 비용 포함")
    print("사용률   평균 ms   p50 ms   p99 ms   이론 ms")
    rows = []
    for rate in (50, 70, 90):
        planned, missed, samples = 0, 0, []
        try:
            with LocalServer(("127.0.0.1", 0), SleepHandler) as server:
                server.service_rng = random.Random(SEED + 200 + rate)
                thread = Thread(target=server.serve_forever, args=(0.05,))
                thread.start()
                try:
                    planned, missed, samples = await http_window(
                        server.server_port, rate, duration,
                        random.Random(SEED + 300 + rate))
                finally:
                    server.shutdown()
                    thread.join()
        except OSError as exc:
            print(f"로컬 서버 오류 (번호 {exc.errno})")

        mean = fmean(samples) if samples else None
        p50 = percentile(samples, 0.50) if samples else None
        p99 = percentile(samples, 0.99) if samples else None
        theory = 1000 / (100 - rate)
        passed = planned > 0 and len(samples) == planned
        rows.append(dict(utilization=rate / 100, planned=planned,
                         succeeded=len(samples), missed=missed,
                         failed=planned - len(samples), mean_ms=mean,
                         p50_ms=p50, p99_ms=p99, theory_ms=theory,
                         passed=passed))
        values = [f"{v:8.1f}" if v is not None else "      --"
                  for v in (mean, p50, p99, theory)]
        print(f"{rate:4}% " + " ".join(values))
        print(f"  예정 {planned}건 중 성공 {len(samples)}건"
              f" (미발송 {missed}건)  {verdict(passed)}")
    return result(all(row["passed"] for row in rows), rows, seconds=duration)


http = await experiment2(QUICK)

In [ ]:
for lang in chart_languages:
    t = labels[lang]
    fig, ax = plt.subplots()
    rows = simulation["rows"]
    loads = [row["rate_qps"] for row in rows]  # μ=100이므로 도착률의 수치가 사용률(%)과 같다.
    grid = [min(loads) + (max(loads) - min(loads)) * i / 300 for i in range(301)]
    ax.plot(grid, [1000 / (100 - x) for x in grid], label=t["theory"])
    ax.scatter(loads, [row["mean_ms"] for row in rows], color=CHART_COLORS[1],
               zorder=3, label=t["sim"])
    valid = [row for row in http["rows"] if row["mean_ms"] is not None]
    ax.scatter([100 * row["utilization"] for row in valid], [row["mean_ms"] for row in valid],
               marker="x", color=CHART_COLORS[2], zorder=4, label=t["http"])
    ax.set(title=t["response"], xlabel=t["util"], ylabel=t["ms"], yscale="log")
    ax.legend()
    note = t["response_note"].format(n=rows[0]["samples"], s=http["seconds"])
    finish_chart(fig, f"{CPU_RUNTIME}, {note}")

### 3. 재시도 증폭

가상 시계에서 매초 처리 한계를 넘는 요청을 넣습니다. 첫 시도와 재시도를 합친 백엔드 요청 수를 정책별로 비교합니다.

In [ ]:
def retry_series(mode):
    capacity, fresh, seconds = 10_000, 10_100, 30
    # 양수는 재시도 차수. 예정 시각마다 다음 차수의 요청을 모은다.
    due = defaultdict(list)
    lottery = random.Random(SEED + 401)
    jitter = random.Random(SEED + 402)
    rows = []
    dropped_total = 0
    for second in range(1, seconds + 1):
        retries = due.pop(second, [])
        dropped = 0
        if mode in ("budget_only", "combined"):
            # r / (fresh + r) <= 0.10. 예산 초과분은 포기한다.
            allowed = fresh // 9
            dropped = max(0, len(retries) - allowed)
            retries = retries[:allowed]
        attempts = [0] * fresh + retries
        # 수락 순서는 무작위. 거절될 위치만 비복원 추출한다.
        rejected = lottery.sample(attempts, max(0, len(attempts) - capacity))
        first_ok = fresh - rejected.count(0)
        for age in rejected:
            if mode in ("unlimited", "budget_only"):
                due[second + 1].append(age + 1)
            elif age == 3:
                dropped += 1
            else:
                delay = jitter.randint(1, 2 ** (age + 1))
                due[second + delay].append(age + 1)
        dropped_total += dropped
        rows.append(dict(second=second, attempts=len(attempts),
                         retries=len(retries), first_ok=first_ok,
                         first_success_rate=first_ok / fresh,
                         succeeded=len(attempts) - len(rejected),
                         dropped=dropped))
    pending = sum(map(len, due.values()))
    accounted = sum(row["succeeded"] for row in rows) + dropped_total + pending
    return dict(rows=rows, pending=pending, dropped=dropped_total,
                conserved=accounted == fresh * seconds)


def experiment3(quick=False):
    heading(3, "재시도 증폭")
    print("한계 10,000/초, 매초 첫 시도 10,100건")
    print("각 칸은 백엔드 요청 수 / 첫 시도 성공률(%)")
    print(" 초       무제한       3회·백오프       예산만        둘 다")
    modes = ("unlimited", "backoff", "budget_only", "combined")
    series = {mode: retry_series(mode) for mode in modes}
    for group in zip(*(series[m]["rows"] for m in modes)):
        cells = [f"{r['attempts']:5}/{100 * r['first_success_rate']:4.1f}"
                 for r in group]
        print(f"{group[0]['second']:3}   " + "   ".join(cells))
    raw, backoff, budget, combined = (series[m]["rows"] for m in modes)
    passed = (all(s["conserved"] for s in series.values())
              and all(r["attempts"] == 10000 + r["second"] * 100 for r in raw)
              and all(10 * r["retries"] <= r["attempts"]
                      for rows in (budget, combined) for r in rows)
              and series["budget_only"]["dropped"] > 0
              and sum(r["attempts"] for r in backoff)
              < sum(r["attempts"] for r in raw))
    print("백오프 지터 1~2·1~4·1~8초, 예산은 매초 전체의 10%")
    print("조건             최종 포기    30초 뒤 재시도 대기")
    for mode, name in zip(modes, ("무제한", "백오프", "예산만", "둘 다")):
        print(f"{name}  {series[mode]['dropped']:12,}"
              f"  {series[mode]['pending']:12,}")
    return result(passed, {m: series[m]["rows"] for m in modes},
                  pending={m: series[m]["pending"] for m in modes},
                  dropped={m: series[m]["dropped"] for m in modes})


retry = experiment3(QUICK)

In [ ]:
for lang in chart_languages:
    t = labels[lang]
    fig, ax = plt.subplots()
    for mode in ("unlimited", "backoff", "budget_only"):
        rows = retry["rows"][mode]
        ax.plot([r["second"] for r in rows], [r["attempts"] for r in rows], label=t[mode])
    ax.axhline(10_000, color=CHART_LIMIT, linestyle="--", linewidth=1)
    ax.set(title=t["retry"], xlabel=t["seconds"], ylabel=t["attempts"])
    ax.legend()
    finish_chart(fig, f"{CPU_RUNTIME}, {t['retry_note']}")

### 4. 느린 요청이 작업자를 점유하는 시간

일부 요청의 처리 시간이 길 때 작업자를 배정받은 비율과 정상 응답을 받은 비율을 가상 시간에서 셉니다.

In [ ]:
def experiment4(quick=False):
    heading(4, "느린 요청의 스레드 점유")
    workers, rate = 1000, 1000
    slow_fraction = 0.05
    mean_hold = (1 - slow_fraction) * 0.1 + slow_fraction * 100
    theory = min(1, workers / (rate * mean_hold))
    warmup, seconds = 1000, (600 if quick else 3600)
    rng = random.Random(SEED + 500)
    busy = []
    rows = []
    # 정수 밀리초 시계로 매 1ms 도착. 자리가 없으면 즉시 거절.
    for block in range(-warmup // 100, seconds // 100):
        admitted = normal = 0
        for now in range(block * 100_000, (block + 1) * 100_000):
            while busy and busy[0] <= now:
                heapq.heappop(busy)
            if len(busy) < workers:
                slow = rng.random() < slow_fraction
                heapq.heappush(busy, now + (100_000 if slow else 100))
                admitted += 1
                normal += int(not slow)
        if block >= 0:
            rows.append(dict(end_s=(block + 1) * 100, offered=rate * 100,
                             admitted=admitted, normal=normal,
                             timed_out=admitted - normal))
    offered = seconds * rate
    measured = sum(r["admitted"] for r in rows) / offered
    normal_rate = sum(r["normal"] for r in rows) / offered
    print("1,000스레드, 초당 1,000건 중 5%가 100초 점유")
    print(f"평균 점유 {mean_hold:.3f}초, 이론 배정률 {theory:.1%}")
    print(f"예열 {warmup:,}초 뒤 {seconds:,}초 측정 (가상 시간)")
    print(f"실측 배정률 {measured:.2%}, 정상 응답률 {normal_rate:.2%}")
    print("배정률에는 타임아웃 포함. 허용 차이 1%p")
    return result(abs(measured - theory) <= 0.01, rows,
                  theory_fraction=theory, admitted_fraction=measured,
                  normal_fraction=normal_rate)


occupancy = experiment4(QUICK)

### 5. 대기열 상한과 부하 차단

같은 도착열에 두 정책을 적용합니다. 처리량은 측정 구간에 완료한 요청으로, p99는 수락한 모든 요청의 응답 시간으로 계산합니다. 표에는 503 거절 비율도 함께 표시합니다.

In [ ]:
def shed_window(arrivals, limit, duration, workers, service):
    # 처리 시간이 일정해 완료 순서가 도착 순서와 같다.
    finishes = deque()
    latency_ms = []
    completed = peak_queue = 0
    for arrival in arrivals:
        while finishes and finishes[0] <= arrival:
            finishes.popleft()
        if limit is not None and len(finishes) >= workers + limit:
            continue
        start = arrival if len(finishes) < workers else finishes[-workers]
        finish = start + service
        finishes.append(finish)
        peak_queue = max(peak_queue, len(finishes) - workers)
        completed += int(finish <= duration)
        latency_ms.append(1000 * (finish - arrival))
    latency_ms.sort()
    rejected = len(arrivals) - len(latency_ms)
    return dict(queue_limit=limit, peak_queue=peak_queue,
                offered=len(arrivals), accepted=len(latency_ms),
                completed_in_window=completed, throughput_qps=completed / duration,
                p99_ms=percentile(latency_ms, 0.99), rejected_503=rejected,
                rejection_rate=rejected / len(arrivals))


def experiment5(quick=False):
    heading(5, "부하 차단과 대기열 상한")
    workers, service, duration = 10, 0.1, 60
    capacity = workers / service
    rng = random.Random(SEED + 600)
    arrivals = []
    arrival = rng.expovariate(1.2 * capacity)
    while arrival < duration:
        arrivals.append(arrival)
        arrival += rng.expovariate(1.2 * capacity)
    rows = [shed_window(arrivals, limit, duration, workers, service)
            for limit in (None, workers // 2)]
    print("작업자 10개 × 초당 10건, 도착률 120/초, 60초")
    print("처리량은 구간 내 완료, p99는 수락한 전원 기준")
    print("대기열   처리량/초   성공 p99 ms   503 비율")
    for row, name in zip(rows, ("무제한", "상한 5")):
        print(f"{name}   {row['throughput_qps']:8.2f}"
              f"   {row['p99_ms']:11.1f}   {row['rejection_rate']:8.2%}")
    unbounded, bounded = rows
    passed = (unbounded["rejected_503"] == 0 and bounded["rejected_503"] > 0
              and bounded["peak_queue"] <= workers // 2
              and bounded["p99_ms"] < unbounded["p99_ms"]
              and all(0 < r["throughput_qps"] <= capacity for r in rows))
    return result(passed, rows, seconds=duration)


shedding = experiment5(QUICK)

In [ ]:
for metric, unit in (("throughput_qps", "qps"), ("p99_ms", "p99")):
    for lang in chart_languages:
        t = labels[lang]
        fig, ax = plt.subplots()
        values = [row[metric] for row in shedding["rows"]]
        bars = ax.bar([t["queue"], t["shed"]], values, width=0.5, color=CHART_COLORS[0])
        ax.bar_label(bars, fmt="%.1f", color=CHART_TEXT, padding=3)
        ax.margins(y=0.15)
        ax.set(title=t[metric], ylabel=t[unit])
        note = t["shed_note"].format(s=shedding["seconds"])
        finish_chart(fig, f"{CPU_RUNTIME}, {note}")